# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Regularization

Control overfitting with Ridge, Lasso, and ElasticNet regression.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error

## The Overfitting Problem

In [ ]:
# Load data
housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Data loaded: {X_train.shape}")

## Ridge Regression (L2 Regularization)

Adds penalty proportional to sum of squared coefficients.

Cost = MSE + α × (sum of squared coefficients)

In [ ]:
# Ridge shrinks all coefficients toward zero
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)
ridge_r2 = r2_score(y_test, y_pred_ridge)

print(f"Ridge (alpha=1.0) - Test R²: {ridge_r2:.4f}")
print(f"\nRidge coefficients (first 5):")
for i in range(5):
    print(f"  Feature {i}: {ridge.coef_[i]:.6f}")

## Lasso Regression (L1 Regularization)

Adds penalty proportional to sum of absolute coefficients.

Cost = MSE + α × (sum of absolute coefficients)

**Key difference:** Lasso drives some coefficients to exactly zero (feature selection).

In [ ]:
# Lasso performs feature selection by zeroing out coefficients
lasso = Lasso(alpha=0.01)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)
lasso_r2 = r2_score(y_test, y_pred_lasso)

print(f"Lasso (alpha=0.01) - Test R²: {lasso_r2:.4f}")
print(f"\nLasso coefficients:")
for i, name in enumerate(X.columns):
    print(f"  {name}: {lasso.coef_[i]:.6f}")

n_zero = (lasso.coef_ == 0).sum()
print(f"\nNumber of zero coefficients: {n_zero} / {len(lasso.coef_)}")

## ElasticNet: Hybrid of Ridge and Lasso

In [ ]:
# ElasticNet = Ridge + Lasso
# l1_ratio: 0 = pure Ridge, 1 = pure Lasso, 0.5 = equal mix

elastic = ElasticNet(alpha=0.01, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)

y_pred_elastic = elastic.predict(X_test_scaled)
elastic_r2 = r2_score(y_test, y_pred_elastic)

print(f"ElasticNet (alpha=0.01, l1_ratio=0.5) - Test R²: {elastic_r2:.4f}")

## Step 3: Effect of Alpha (Regularization Strength)

In [ ]:
# As alpha increases, coefficients shrink (less complex model)
alphas = np.logspace(-3, 2, 20)  # 0.001 to 100

ridge_r2_train = []
ridge_r2_test = []

for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    
    train_r2 = ridge.score(X_train_scaled, y_train)
    test_r2 = ridge.score(X_test_scaled, y_test)
    
    ridge_r2_train.append(train_r2)
    ridge_r2_test.append(test_r2)

# Plot
plt.figure(figsize=(10, 6))
plt.semilogx(alphas, ridge_r2_train, label='Training R²', marker='o')
plt.semilogx(alphas, ridge_r2_test, label='Test R²', marker='s')
plt.xlabel('Alpha (Regularization Strength)')
plt.ylabel('R²')
plt.title('Effect of Regularization on Train/Test Performance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nAs alpha increases:")
print(f"  - Coefficients shrink (simpler model)")
print(f"  - Training R² decreases (worse on training data)")
print(f"  - Test R² may increase (less overfitting)")

## Step 4: Model Comparison

In [ ]:
# Compare all methods
print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

# Linear Regression (no regularization)
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_r2 = lr.score(X_test_scaled, y_test)
print(f"\nLinear Regression:")
print(f"  Test R²: {lr_r2:.4f}")

# Ridge
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
ridge_r2 = ridge.score(X_test_scaled, y_test)
print(f"\nRidge (alpha=1.0):")
print(f"  Test R²: {ridge_r2:.4f}")

# Lasso  
lasso = Lasso(alpha=0.01)
lasso.fit(X_train_scaled, y_train)
lasso_r2 = lasso.score(X_test_scaled, y_test)
print(f"\nLasso (alpha=0.01):")
print(f"  Test R²: {lasso_r2:.4f}")

# ElasticNet
elastic = ElasticNet(alpha=0.01, l1_ratio=0.5)
elastic.fit(X_train_scaled, y_train)
elastic_r2 = elastic.score(X_test_scaled, y_test)
print(f"\nElasticNet (alpha=0.01):")
print(f"  Test R²: {elastic_r2:.4f}")